# 01 — Data Exploration
**NY State Least-Cost Renewable Energy Pathway Analyzer**

This notebook explores:
1. The synthetic NYISO hourly demand profile (calibration to 2023 Gold Book)
2. NREL ATB 2023 Moderate cost assumptions
3. Technology learning curve projections 2023–2040
4. Data quality checks


In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from data.generate_data import generate_nyiso_demand, generate_nrel_atb, generate_price_projections
from config import TECH_COLORS, SEASON_COLORS

demand_df = generate_nyiso_demand()
atb_df = generate_nrel_atb()
proj_df = generate_price_projections()

print(f"Demand rows: {len(demand_df):,} hours")
print(f"Annual energy: {demand_df['demand_mw'].sum()/1e6:.1f} TWh")
print(f"Summer peak:   {demand_df['demand_gw'].max():.2f} GW")
print(f"Average load:  {demand_df['demand_gw'].mean():.2f} GW")
print(f"Load factor:   {demand_df['demand_gw'].mean()/demand_df['demand_gw'].max():.1%}")
demand_df.head()

## 1. Full 8,760-Hour Demand Timeseries

In [ ]:
fig = go.Figure(go.Scatter(
    x=demand_df["timestamp"], y=demand_df["demand_gw"],
    mode="lines", line=dict(color="#1D9E75", width=0.7),
    fill="tozeroy", fillcolor="rgba(29,158,117,0.1)",
))
fig.update_layout(
    title="NYISO Synthetic Hourly Demand (2023)",
    xaxis_title="Date", yaxis_title="Demand (GW)",
    height=350, plot_bgcolor="white", yaxis=dict(gridcolor="#f0f0f0"),
    xaxis=dict(rangeslider_visible=True, rangeslider_thickness=0.05),
)
fig.show()

## 2. Monthly and Seasonal Patterns

In [ ]:
month_names = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]
monthly = demand_df.groupby("month")["demand_gw"].agg(["mean","max","min"]).reset_index()
monthly["month_name"] = monthly["month"].apply(lambda m: month_names[m-1])

fig = make_subplots(rows=1, cols=2, subplot_titles=["Monthly Average Demand", "Monthly Peak Demand"])
fig.add_trace(go.Bar(x=monthly["month_name"], y=monthly["mean"],
    marker_color="#1D9E75", name="Average"), row=1, col=1)
fig.add_trace(go.Bar(x=monthly["month_name"], y=monthly["max"],
    marker_color="#E76F51", name="Peak"), row=1, col=2)
fig.update_layout(height=320, plot_bgcolor="white", showlegend=False,
                  yaxis=dict(gridcolor="#f0f0f0"), yaxis2=dict(gridcolor="#f0f0f0"))
fig.show()

In [ ]:
season_map = {1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",
              6:"Summer",7:"Summer",8:"Summer",9:"Fall",10:"Fall",11:"Fall",12:"Winter"}
demand_df2 = demand_df.copy()
demand_df2["season"] = demand_df2["month"].map(season_map)

fig = go.Figure()
for season, color in SEASON_COLORS.items():
    profile = demand_df2[demand_df2["season"]==season].groupby("hour_of_day")["demand_gw"].mean()
    fig.add_trace(go.Scatter(x=profile.index, y=profile.values,
        name=season, mode="lines+markers", line=dict(color=color, width=2.5)))
fig.update_layout(
    title="Average Hourly Demand by Season",
    xaxis_title="Hour of Day", yaxis_title="GW",
    xaxis=dict(tickvals=list(range(0,24,3)), ticktext=[f"{h:02d}:00" for h in range(0,24,3)]),
    height=320, plot_bgcolor="white", yaxis=dict(gridcolor="#f0f0f0"),
    legend=dict(orientation="h", y=-0.25),
)
fig.show()

## 3. NREL ATB 2023 Cost Assumptions

In [ ]:
print(atb_df[["technology","capital_cost_per_kw","fixed_om_per_kw_yr",
             "var_om_per_mwh","capacity_factor","lcoe_per_mwh","lifetime_yr"]].to_string(index=False))

In [ ]:
fig = go.Figure(go.Bar(
    x=atb_df["technology"], y=atb_df["lcoe_per_mwh"],
    marker_color=[TECH_COLORS[t] for t in atb_df["technology"]],
    text=[f"${v:.0f}" for v in atb_df["lcoe_per_mwh"]],
    textposition="outside",
))
fig.add_hline(y=76, line_dash="dot", line_color="red", annotation_text="US avg retail $76/MWh")
fig.update_layout(
    title="Technology LCOEs (NREL ATB 2023 Moderate)",
    yaxis_title="LCOE ($/MWh)", height=380,
    plot_bgcolor="white", yaxis=dict(gridcolor="#f0f0f0"), showlegend=False,
)
fig.show()

## 4. Learning Curve Projections 2023–2040

In [ ]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Capital Cost Decline ($/kW)", "LCOE Projection ($/MWh)"])
fig.add_trace(go.Scatter(x=proj_df["year"], y=proj_df["solar_capital_per_kw"],
    name="Solar PV", mode="lines+markers", line=dict(color="#FFB703", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=proj_df["year"], y=proj_df["onshore_wind_capital_per_kw"],
    name="Onshore Wind", mode="lines+markers", line=dict(color="#219EBC", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=proj_df["year"], y=proj_df["offshore_wind_capital_per_kw"],
    name="Offshore Wind", mode="lines+markers", line=dict(color="#0077B6", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=proj_df["year"], y=proj_df["battery_capital_per_kw"],
    name="Battery", mode="lines+markers", line=dict(color="#8338EC", width=2, dash="dash")), row=1, col=1)

fig.add_trace(go.Scatter(x=proj_df["year"], y=proj_df["solar_lcoe_per_mwh"],
    name="Solar", mode="lines+markers", line=dict(color="#FFB703", width=2), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=proj_df["year"], y=proj_df["onshore_wind_lcoe_per_mwh"],
    name="Onshore", mode="lines+markers", line=dict(color="#219EBC", width=2), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=proj_df["year"], y=proj_df["offshore_wind_lcoe_per_mwh"],
    name="Offshore", mode="lines+markers", line=dict(color="#0077B6", width=2), showlegend=False), row=1, col=2)

fig.update_layout(height=380, plot_bgcolor="white",
                  legend=dict(orientation="h", y=-0.25))
fig.update_xaxes(gridcolor="#f0f0f0")
fig.update_yaxes(gridcolor="#f0f0f0")
fig.show()

## 5. Data Quality Checks

In [ ]:
print("=== Demand data quality ===")
print(f"  Missing values: {demand_df.isnull().sum().sum()}")
print(f"  Annual energy:  {demand_df['demand_mw'].sum()/1e6:.1f} TWh  (target: ~154 TWh)")
print(f"  Peak demand:    {demand_df['demand_gw'].max():.2f} GW (target: ~28 GW)")
print(f"  Min demand:     {demand_df['demand_gw'].min():.2f} GW")
print(f"  Hours < 12 GW:  {(demand_df['demand_gw'] < 12).sum()}")
print()
print("=== ATB data quality ===")
print(f"  Missing values: {atb_df.isnull().sum().sum()}")
print(f"  Negative costs: {(atb_df[['capital_cost_per_kw','lcoe_per_mwh']] < 0).any().any()}")
print(f"  Technologies:   {len(atb_df)}")
print(atb_df[["technology","lcoe_per_mwh"]].to_string(index=False))